# HH.ru ELT Pipeline

Инкрементальная загрузка вакансий с hh.ru в PostgreSQL с сохранением в файловую систему (Parquet).

Источник данных — каталог вакансий `https://hh.ru/vacancies/voditel` (веб-скрейпинг, JSON из `template#HH-Lux-InitialState`).

Возможности:
- Скрейпинг каталога вакансий по страницам без официального API
- Инкрементальная загрузка: INSERT новых, UPDATE изменённых, мягкое DELETE удалённых
- Уникальный `etl_id` на каждый запуск, сквозное логирование
- Retry с экспоненциальным backoff, задержка между страницами
- Конфигурация через Hydra + переменные окружения
- Сохранение в Parquet (snappy) в data_lake/

## 1. Импорты и конфигурация

In [1]:
import logging
import uuid
import hashlib
import json
import os
import time
from pathlib import Path
from urllib.parse import urlencode

import requests
from bs4 import BeautifulSoup
import tenacity
import polars as pl
import pendulum
from sqlalchemy import create_engine, text
from tqdm.auto import tqdm
import hydra
from omegaconf import OmegaConf

In [2]:
# Hydra может быть инициализирован только один раз за сессию
try:
    hydra.initialize(version_base=None, config_path='conf')
except Exception:
    pass
conf = hydra.compose(config_name='config')
print(OmegaConf.to_yaml(conf))

db:
  host: ${oc.env:DB_HOST,localhost}
  port: ${oc.env:DB_PORT,5432}
  name: ${oc.env:DB_NAME,etl_db}
  user: ${oc.env:DB_USER,etl}
  password: ${oc.env:DB_PASSWORD,password}
  schema: etl
hh_scraper:
  catalog_url: https://hh.ru/vacancies/voditel
  area: 1
  items_on_page: 100
  order_by: publication_time
  enable_snippets: 'true'
  user_agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:133.0) Gecko/20100101 Firefox/133
  request_delay: 1.5
data_lake:
  base_path: ./data_lake
  compression: snappy
logging:
  level: DEBUG
  file: logs/hh_etl.log



## 2. ETL ID и логирование

In [3]:
# Генерируем уникальный ID для этого запуска
ETL_ID = str(uuid.uuid4())
RUN_STARTED_AT = pendulum.now('UTC')
print(f'ETL_ID: {ETL_ID}')
print(f'Started at: {RUN_STARTED_AT}')

ETL_ID: bd8e50f1-20b5-4969-a4f8-100a1a657f35
Started at: 2026-03-29 13:17:03.142690+00:00


In [4]:
Path('logs').mkdir(exist_ok=True)

log_formatter = logging.Formatter(
    f'%(asctime)s | etl_id={ETL_ID} | %(name)s | %(levelname)s | %(message)s'
)

file_handler = logging.FileHandler(conf.logging.file, mode='a', encoding='utf-8')
file_handler.setFormatter(log_formatter)

console_handler = logging.StreamHandler()
console_handler.setFormatter(log_formatter)

logger = logging.getLogger('hh_etl')
logger.setLevel(conf.logging.level)
logger.handlers.clear()
logger.addHandler(file_handler)
logger.addHandler(console_handler)
logger.propagate = False

# Подавляем шум от sqlalchemy и urllib3
logging.getLogger('sqlalchemy.engine').setLevel(logging.WARNING)
logging.getLogger('urllib3').propagate = False

logger.info('ETL pipeline starting')

2026-03-29 16:17:03,161 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | INFO | ETL pipeline starting


## 3. Retry-декоратор

In [5]:
def _before_sleep(retry_state):
    logger.warning(
        f'Attempt #{retry_state.attempt_number} failed: {retry_state.outcome.exception()}. '
        f'Retrying in {retry_state.upcoming_sleep:.1f}s'
    )

def _on_giveup(retry_state):
    logger.error(
        f'All {retry_state.attempt_number} attempts failed. '
        f'Last error: {retry_state.outcome.exception()}'
    )
    return retry_state.outcome.exception()

http_retry = tenacity.retry(
    stop=tenacity.stop_after_attempt(5),
    wait=tenacity.wait_exponential(multiplier=1, min=1, max=30),
    retry=tenacity.retry_if_exception_type((requests.HTTPError, requests.ConnectionError, requests.Timeout)),
    before_sleep=_before_sleep,
    retry_error_callback=_on_giveup,
    reraise=False,
)

## 4. HH.ru — веб-скрейпинг

In [6]:
SCRAPER_HEADERS = {
    'User-Agent': conf.hh_scraper.user_agent,
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8',
}


@http_retry
def _fetch_page(url: str) -> bytes:
    """Загружает HTML-страницу с retry."""
    response = requests.get(url, headers=SCRAPER_HEADERS, timeout=20)
    response.raise_for_status()
    return response.content


def _parse_initial_state(html: bytes) -> dict:
    """Извлекает JSON из тега template#HH-Lux-InitialState."""
    soup = BeautifulSoup(html, 'lxml')
    tag = soup.select_one('template#HH-Lux-InitialState')
    if not tag:
        raise ValueError('HH-Lux-InitialState template not found on page')
    return json.loads(tag.text)


def _build_url(page: int) -> str:
    """Строит URL страницы каталога с GET-параметрами."""
    params = {
        'area': conf.hh_scraper.area,
        'items_on_page': conf.hh_scraper.items_on_page,
        'order_by': conf.hh_scraper.order_by,
        'enable_snippets': conf.hh_scraper.enable_snippets,
        'page': page,
    }
    return f"{conf.hh_scraper.catalog_url}?{urlencode(params)}"


def extract_all_vacancies() -> list[dict]:
    """Скрейпит все страницы каталога и возвращает список сырых вакансий."""
    logger.info(f'Starting extraction from {conf.hh_scraper.catalog_url}')

    # Первая страница — определяем количество страниц
    html0 = _fetch_page(_build_url(0))
    data0 = _parse_initial_state(html0)

    paging = data0['vacancySearchResult']['paging']
    if paging is None:
        pages_count = 1
    elif paging.get('lastPage') is None:
        pages_count = paging['pages'][-1]['page'] + 1
    else:
        pages_count = paging['lastPage']['page'] + 1

    total_found = (data0.get('searchCounts') or {}).get('value', '?')
    logger.info(f'Found ~{total_found} vacancies, {pages_count} pages to fetch')

    all_vacancies = list(data0['vacancySearchResult']['vacancies'])

    for page_idx in tqdm(range(1, pages_count), desc='Pages'):
        time.sleep(conf.hh_scraper.request_delay)
        html = _fetch_page(_build_url(page_idx))
        data = _parse_initial_state(html)
        page_vacancies = data['vacancySearchResult']['vacancies']
        all_vacancies.extend(page_vacancies)
        logger.debug(f'Page {page_idx}/{pages_count - 1}: +{len(page_vacancies)} vacancies')

    logger.info(f'Extracted {len(all_vacancies)} vacancies total')
    return all_vacancies

## 5. Трансформация

In [7]:
def _safe(d: dict, *keys, default=None):
    """Безопасное вложенное получение значения из dict."""
    for k in keys:
        if not isinstance(d, dict):
            return default
        d = d.get(k)
        if d is None:
            return default
    return d


def _parse_dt(val) -> str | None:
    """Парсит ISO-дату или объект {$: '...'} в строку UTC."""
    if not val:
        return None
    if isinstance(val, dict):
        val = val.get('$') or val.get('milliseconds')
        if not val:
            return None
    try:
        return pendulum.parse(str(val)).isoformat()
    except Exception:
        return None


def _row_hash(vacancy_id: str, name: str, last_change_time) -> str:
    """SHA-256 хэш для обнаружения изменений вакансии."""
    raw = f'{vacancy_id}|{name}|{last_change_time}'
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()


def parse_vacancy(item: dict, extracted_dttm: str) -> dict:
    """
    Разбирает сырой элемент из vacancySearchResult['vacancies'] в плоскую запись.
    Структура соответствует данным, возвращаемым HH-Lux-InitialState при скрейпинге.
    """
    vid = str(item.get('vacancyId', ''))
    name = item.get('name', '')

    company = item.get('company') or {}
    area = item.get('area') or {}
    snippet = item.get('snippet') or {}

    # Зарплата — в скрейп-данных может лежать в разных местах
    compensation = item.get('compensation') or item.get('salary') or {}
    salary_from = _safe(compensation, 'from') or _safe(compensation, 'minimum')
    salary_to = _safe(compensation, 'to') or _safe(compensation, 'maximum')
    salary_currency = _safe(compensation, 'currency') or _safe(compensation, 'currencyCode')

    last_change_time = _parse_dt(item.get('lastChangeTime'))

    row = {
        'etl_id': ETL_ID,
        'extracted_dttm': extracted_dttm,
        'row_hash': _row_hash(vid, name, last_change_time),

        'vacancy_id': int(vid) if vid.isdigit() else None,
        'name': name,
        'alternate_url': f'https://hh.ru/vacancy/{vid}' if vid.isdigit() else None,
        'is_adv': item.get('@isAdv') == 'true',

        # Работодатель
        'employer_id': int(company['id']) if str(company.get('id', '')).isdigit() else None,
        'employer_name': company.get('name'),
        'employer_visible_name': company.get('visibleName'),

        # Регион
        'area_id': int(area['id']) if str(area.get('id', '')).isdigit() else None,
        'area_name': area.get('name'),

        # Зарплата
        'salary_from': salary_from,
        'salary_to': salary_to,
        'salary_currency': salary_currency,

        # Даты
        'published_at': _parse_dt(item.get('publicationTime')),
        'last_change_time': last_change_time,
        'created_at': _parse_dt(item.get('creationTime')),

        # Статистика откликов
        'responses_count': item.get('responsesCount'),
        'total_responses_count': item.get('totalResponsesCount'),

        # Сниппет (requirement, responsibility) и полный JSON
        'snippet': json.dumps(snippet, ensure_ascii=False),
        'raw_json': json.dumps(item, ensure_ascii=False),
    }
    return row


def transform(raw_vacancies: list[dict]) -> pl.DataFrame:
    """Преобразует список сырых вакансий в Polars DataFrame."""
    extracted_dttm = pendulum.now('UTC').isoformat()
    rows = [parse_vacancy(item, extracted_dttm) for item in raw_vacancies]
    df = pl.DataFrame(rows, infer_schema_length=len(rows) or 1)

    # HH повторяет promoted-вакансии на нескольких страницах — дедуплицируем
    before = len(df)
    df = df.unique(subset=['vacancy_id'], keep='first', maintain_order=True)
    dupes = before - len(df)
    if dupes:
        logger.info(f'Removed {dupes} duplicate vacancy_id rows (promoted vacancies)')

    logger.info(f'Transformed {len(df)} records')
    return df

## 6. БД — инициализация

In [8]:
DATABASE_URL = (
    f"postgresql+psycopg2://{conf.db.user}:{conf.db.password}"
    f"@{conf.db.host}:{conf.db.port}/{conf.db.name}"
)
db = create_engine(DATABASE_URL, pool_pre_ping=True)
logger.info(f'DB engine created: {conf.db.host}:{conf.db.port}/{conf.db.name}')

2026-03-29 16:17:03,321 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | INFO | DB engine created: localhost:5432/etl_db


In [9]:
DDL = """
CREATE SCHEMA IF NOT EXISTS etl;

CREATE TABLE IF NOT EXISTS etl.hh_vacancies (
    -- ETL-метаданные
    etl_id                TEXT        NOT NULL,
    created_dttm          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    updated_dttm          TIMESTAMPTZ,
    deleted_dttm          TIMESTAMPTZ,
    is_deleted            BOOLEAN     NOT NULL DEFAULT FALSE,
    extracted_dttm        TIMESTAMPTZ NOT NULL,
    row_hash              TEXT        NOT NULL,

    -- Ключ
    vacancy_id            BIGINT      PRIMARY KEY,
    name                  TEXT        NOT NULL,
    alternate_url         TEXT,
    is_adv                BOOLEAN,

    -- Работодатель
    employer_id           BIGINT,
    employer_name         TEXT,
    employer_visible_name TEXT,

    -- Регион
    area_id               INT,
    area_name             TEXT,

    -- Зарплата
    salary_from           INT,
    salary_to             INT,
    salary_currency       TEXT,

    -- Даты
    published_at          TIMESTAMPTZ,
    last_change_time      TIMESTAMPTZ,
    created_at            TIMESTAMPTZ,

    -- Статистика откликов
    responses_count       INT,
    total_responses_count INT,

    -- Контент
    snippet               JSONB,
    raw_json              JSONB
);

CREATE TABLE IF NOT EXISTS etl.hh_etl_runs (
    etl_id              TEXT        PRIMARY KEY,
    started_at          TIMESTAMPTZ NOT NULL,
    finished_at         TIMESTAMPTZ,
    status              TEXT        NOT NULL DEFAULT 'running',
    vacancies_found     INT,
    vacancies_inserted  INT,
    vacancies_updated   INT,
    vacancies_deleted   INT,
    error_message       TEXT
);
"""

with db.connect() as con:
    con.execute(text(DDL))
    con.commit()

logger.info('Schema and tables ensured')

2026-03-29 16:17:03,360 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | INFO | Schema and tables ensured


## 7. Регистрация ETL-запуска

In [10]:
with db.connect() as con:
    con.execute(text("""
        INSERT INTO etl.hh_etl_runs (etl_id, started_at, status)
        VALUES (:etl_id, :started_at, 'running')
    """), {'etl_id': ETL_ID, 'started_at': RUN_STARTED_AT.isoformat()})
    con.commit()

logger.info(f'ETL run registered: etl_id={ETL_ID}')

2026-03-29 16:17:03,378 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | INFO | ETL run registered: etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35


## 8. Extract

In [11]:
raw_vacancies = extract_all_vacancies()

2026-03-29 16:17:03,390 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | INFO | Starting extraction from https://hh.ru/vacancies/voditel
2026-03-29 16:17:04,697 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | INFO | Found ~4711 vacancies, 20 pages to fetch


Pages:   0%|          | 0/19 [00:00<?, ?it/s]

2026-03-29 16:17:07,629 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | DEBUG | Page 1/19: +103 vacancies
2026-03-29 16:17:10,826 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | DEBUG | Page 2/19: +103 vacancies
2026-03-29 16:17:13,610 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | DEBUG | Page 3/19: +103 vacancies
2026-03-29 16:17:16,508 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | DEBUG | Page 4/19: +103 vacancies
2026-03-29 16:17:19,725 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | DEBUG | Page 5/19: +103 vacancies
2026-03-29 16:17:22,934 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | DEBUG | Page 6/19: +103 vacancies
2026-03-29 16:17:25,893 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | DEBUG | Page 7/19: +103 vacancies
2026-03-29 16:17:29,320 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | DEBUG | Page 8/19: +103 vacancies
2026-03-29 16:17:33,356 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl |

## 9. Transform

In [12]:
df = transform(raw_vacancies)
df.head(3)

2026-03-29 16:18:06,694 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | INFO | Removed 17 duplicate vacancy_id rows (promoted vacancies)
2026-03-29 16:18:06,696 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | INFO | Transformed 2042 records


etl_id,extracted_dttm,row_hash,vacancy_id,name,alternate_url,is_adv,employer_id,employer_name,employer_visible_name,area_id,area_name,salary_from,salary_to,salary_currency,published_at,last_change_time,created_at,responses_count,total_responses_count,snippet,raw_json
str,str,str,i64,str,str,bool,i64,str,str,null,str,i64,i64,str,str,str,str,i64,i64,str,str
"""bd8e50f1-20b5-4969-a4f8-100a1a…","""2026-03-29T13:18:06.142171+00:…","""97cbfd2250bb81af62cdb4ee1a5d70…",130419170,"""Водитель категории Е (Коломна)""","""https://hh.ru/vacancy/13041917…",false,49357,"""МАГНИТ, Розничная сеть""","""МАГНИТ, Розничная сеть. Трансп…",null,"""Москва""",172000,247500,"""RUR""","""2026-03-24T07:05:09.575000+03:…","""2026-03-28T04:07:42.459000+03:…","""2026-02-13T08:47:40.639000+03:…",16,74,"""{""req"": ""У вас есть действующе…","""{""@workSchedule"": ""flexible"", …"
"""bd8e50f1-20b5-4969-a4f8-100a1a…","""2026-03-29T13:18:06.142171+00:…","""5a40b293e3a6a382e58ceaf89ef84d…",131405375,"""Водитель-курьер в Озон Фреш (м…","""https://hh.ru/vacancy/13140537…",false,2180,"""Ozon""","""Ozon fresh""",null,"""Москва""",150000,270000,"""RUR""","""2026-03-20T09:44:57.596000+03:…","""2026-03-27T09:45:04.630000+03:…","""2026-03-20T09:44:57.596000+03:…",24,30,"""{""req"": null, ""resp"": ""Оформле…","""{""@workSchedule"": ""fullDay"", ""…"
"""bd8e50f1-20b5-4969-a4f8-100a1a…","""2026-03-29T13:18:06.142171+00:…","""7b3d823fc660bac46fabbdfa450c65…",131603142,"""Водитель-экспедитор Газель НЕК…","""https://hh.ru/vacancy/13160314…",false,2435936,"""СОРТАМЕНТ""","""СОРТАМЕНТ""",null,"""Москва""",100000,140000,"""RUR""","""2026-03-27T10:45:16.920000+03:…","""2026-03-27T10:57:36.946000+03:…","""2026-03-27T10:45:16.920000+03:…",12,14,"""{""req"": ""Наличие действующего …","""{""@workSchedule"": ""fullDay"", ""…"


## 10. Load — инкрементальная загрузка в PostgreSQL

In [13]:
def load_incremental(df: pl.DataFrame) -> dict:
    """
    Инкрементальная загрузка:
    - INSERT новых вакансий
    - UPDATE изменённых (по row_hash)
    - Мягкий DELETE вакансий, которых нет в текущей выгрузке
    """
    stats = {'inserted': 0, 'updated': 0, 'deleted': 0}

    if df.is_empty():
        logger.warning('Empty dataframe, nothing to load')
        return stats

    current_ids = df['vacancy_id'].drop_nulls().to_list()

    with db.connect() as con:
        # --- Получаем существующие активные записи ---
        existing = pl.read_database(
            "SELECT vacancy_id, row_hash FROM etl.hh_vacancies WHERE is_deleted = FALSE",
            con,
        )
        existing_ids = set(existing['vacancy_id'].to_list()) if not existing.is_empty() else set()
        existing_hashes = (
            dict(zip(existing['vacancy_id'].to_list(), existing['row_hash'].to_list()))
            if not existing.is_empty() else {}
        )

        # --- Разбиваем на NEW / CHANGED ---
        new_rows = df.filter(~pl.col('vacancy_id').is_in(list(existing_ids)))
        changed_rows = df.filter(
            pl.col('vacancy_id').is_in(list(existing_ids))
            & pl.col('vacancy_id').map_elements(
                lambda vid: existing_hashes.get(vid, '') != df.filter(pl.col('vacancy_id') == vid)['row_hash'][0],
                return_dtype=pl.Boolean
            )
        )

        # --- INSERT новых ---
        if not new_rows.is_empty():
            _bulk_insert(new_rows, con)
            stats['inserted'] = len(new_rows)
            logger.info(f'Inserted {stats["inserted"]} new vacancies')

        # --- UPDATE изменённых ---
        if not changed_rows.is_empty():
            _bulk_update(changed_rows, con)
            stats['updated'] = len(changed_rows)
            logger.info(f'Updated {stats["updated"]} changed vacancies')

        # --- Мягкое DELETE ---
        deleted_ids = existing_ids - set(current_ids)
        if deleted_ids:
            con.execute(text("""
                UPDATE etl.hh_vacancies
                SET is_deleted = TRUE,
                    deleted_dttm = NOW(),
                    etl_id = :etl_id
                WHERE vacancy_id = ANY(:ids)
                  AND is_deleted = FALSE
            """), {'etl_id': ETL_ID, 'ids': list(deleted_ids)})
            stats['deleted'] = len(deleted_ids)
            logger.info(f'Soft-deleted {stats["deleted"]} vacancies no longer on HH')

        con.commit()

    return stats


def _row_to_params(row: dict) -> dict:
    """Конвертирует строку DataFrame в параметры для SQL."""
    return {
        k: (v.isoformat() if hasattr(v, 'isoformat') else v)
        for k, v in row.items()
    }


# Колонки соответствуют полям таблицы etl.hh_vacancies (кроме created_dttm/updated_dttm/deleted_dttm)
INSERT_COLS = [
    'etl_id', 'extracted_dttm', 'row_hash',
    'vacancy_id', 'name', 'alternate_url', 'is_adv',
    'employer_id', 'employer_name', 'employer_visible_name',
    'area_id', 'area_name',
    'salary_from', 'salary_to', 'salary_currency',
    'published_at', 'last_change_time', 'created_at',
    'responses_count', 'total_responses_count',
    'snippet',
    'raw_json',
]

INSERT_SQL = f"""
    INSERT INTO etl.hh_vacancies ({', '.join(INSERT_COLS)})
    VALUES ({', '.join(':' + c for c in INSERT_COLS)})
    ON CONFLICT (vacancy_id) DO NOTHING
"""

UPDATE_COLS = [c for c in INSERT_COLS if c not in ('vacancy_id', 'etl_id')]
UPDATE_SQL = f"""
    UPDATE etl.hh_vacancies
    SET updated_dttm = NOW(),
        etl_id = :etl_id,
        {', '.join(f'{c} = :{c}' for c in UPDATE_COLS)}
    WHERE vacancy_id = :vacancy_id
"""


def _bulk_insert(rows_df: pl.DataFrame, con):
    params = [_row_to_params(r) for r in rows_df.select(INSERT_COLS).to_dicts()]
    con.execute(text(INSERT_SQL), params)


def _bulk_update(rows_df: pl.DataFrame, con):
    params = [_row_to_params(r) for r in rows_df.select(INSERT_COLS).to_dicts()]
    con.execute(text(UPDATE_SQL), params)

In [14]:
stats = load_incremental(df)
print(f"Inserted: {stats['inserted']}, Updated: {stats['updated']}, Deleted: {stats['deleted']}")

2026-03-29 16:18:08,914 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | INFO | Inserted 2042 new vacancies


Inserted: 2042, Updated: 0, Deleted: 0


## 11. Сохранение в Data Lake (Parquet + Snappy)

In [15]:
def save_to_parquet(df: pl.DataFrame) -> Path:
    """
    Сохраняет DataFrame в партиционированный Parquet (год/месяц/день).
    Сжатие: snappy (быстро) или zstd (компактнее).
    """
    now = pendulum.now('UTC')
    partition_path = (
        Path(conf.data_lake.base_path)
        / 'hh_vacancies'
        / f'year={now.year}'
        / f'month={now.month:02d}'
        / f'day={now.day:02d}'
    )
    partition_path.mkdir(parents=True, exist_ok=True)

    file_path = partition_path / f'{ETL_ID}.parquet'

    # Исключаем raw_json из parquet (экономия места), храним отдельно
    df_parquet = df.drop('raw_json')
    df_parquet.write_parquet(
        file_path,
        compression=conf.data_lake.compression,
        use_pyarrow=True,
    )

    size_mb = file_path.stat().st_size / 1024 / 1024
    logger.info(f'Saved {len(df)} records to {file_path} ({size_mb:.2f} MB, {conf.data_lake.compression})')
    return file_path


parquet_path = save_to_parquet(df)
print(f'Parquet saved: {parquet_path}')

2026-03-29 16:18:09,155 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | INFO | Saved 2042 records to data_lake/hh_vacancies/year=2026/month=03/day=29/bd8e50f1-20b5-4969-a4f8-100a1a657f35.parquet (0.91 MB, snappy)


Parquet saved: data_lake/hh_vacancies/year=2026/month=03/day=29/bd8e50f1-20b5-4969-a4f8-100a1a657f35.parquet


## 12. Финализация ETL-запуска

In [16]:
RUN_FINISHED_AT = pendulum.now('UTC')
duration_sec = (RUN_FINISHED_AT - RUN_STARTED_AT).total_seconds()

with db.connect() as con:
    con.execute(text("""
        UPDATE etl.hh_etl_runs
        SET finished_at         = :finished_at,
            status              = 'success',
            vacancies_found     = :found,
            vacancies_inserted  = :inserted,
            vacancies_updated   = :updated,
            vacancies_deleted   = :deleted
        WHERE etl_id = :etl_id
    """), {
        'etl_id': ETL_ID,
        'finished_at': RUN_FINISHED_AT.isoformat(),
        'found': len(df),
        'inserted': stats['inserted'],
        'updated': stats['updated'],
        'deleted': stats['deleted'],
    })
    con.commit()

logger.info(
    f'ETL run finished: etl_id={ETL_ID} | '
    f'duration={duration_sec:.1f}s | '
    f'found={len(df)} | inserted={stats["inserted"]} | '
    f'updated={stats["updated"]} | deleted={stats["deleted"]}'
)

2026-03-29 16:18:09,170 | etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | hh_etl | INFO | ETL run finished: etl_id=bd8e50f1-20b5-4969-a4f8-100a1a657f35 | duration=66.0s | found=2042 | inserted=2042 | updated=0 | deleted=0


## 13. Проверочный запрос

In [17]:
with db.connect() as con:
    summary = pl.read_database("""
        SELECT
            COUNT(*)                                   AS total,
            COUNT(*) FILTER (WHERE NOT is_deleted)     AS active,
            COUNT(*) FILTER (WHERE is_deleted)         AS deleted,
            MAX(extracted_dttm)                        AS last_extracted
        FROM etl.hh_vacancies
    """, con)
summary

total,active,deleted,last_extracted
i64,i64,i64,"datetime[μs, UTC]"
2042,2042,0,2026-03-29 13:18:06.142171 UTC


In [18]:
with db.connect() as con:
    runs = pl.read_database("""
        SELECT etl_id, started_at, finished_at, status,
               vacancies_found, vacancies_inserted, vacancies_updated, vacancies_deleted
        FROM etl.hh_etl_runs
        ORDER BY started_at DESC
        LIMIT 10
    """, con)
runs

etl_id,started_at,finished_at,status,vacancies_found,vacancies_inserted,vacancies_updated,vacancies_deleted
str,"datetime[μs, UTC]","datetime[μs, UTC]",str,i64,i64,i64,i64
"""bd8e50f1-20b5-4969-a4f8-100a1a…",2026-03-29 13:17:03.142690 UTC,2026-03-29 13:18:09.164151 UTC,"""success""",2042,2042,0,0
"""2426bcf7-7278-4aa4-a9aa-8a6596…",2026-03-29 13:11:11.912634 UTC,null,"""running""",null,null,null,null
